# M3 일반 관계 밖 CLV 추가 후보 엣지 — Dunnhumby 사전 집합 진단

일반 신규상품 후보 100개를 보존하고, historical CLV proxy가 일반 관계 밖에서 고른 후보 20개만 추가하는 마지막 M3 구조입니다. 이 노트북은 이미 본 `DAY 684~690`에서 **학습 없이 후보 집합만 진단**합니다. 성능 평가·checkpoint 선택·holdout 생성은 하지 않으며, `DAY 705~711`도 열지 않습니다.


## 1. 검증된 소스와 Drive 연결


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = 'c81b3643d8341b2b075ebf53789fc88298527226'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run([
    'git', 'clone', '-q',
    'https://github.com/jung-un/clv-m2-lightgcn-runner.git',
    str(repo),
], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
for name in list(sys.modules):
    if name.startswith(('lightgcn_clv', 'clv_m3', 'clv_run_state')):
        sys.modules.pop(name, None)
importlib.invalidate_caches()
print('Pinned execution source:', actual_sha)


## 2. 고정 설계와 평가 차단 확인


In [ ]:
import json
import lightgcn_clv_m3_clv_conditioned_candidate_item as candidate_runner
import lightgcn_clv_m3_supplemental_candidate_diagnostic as diagnostic

candidate_runner = importlib.reload(candidate_runner)
diagnostic = importlib.reload(diagnostic)
assert candidate_runner.SUPPLEMENTAL_CODE_VERSION == 'm3-clv-supplemental-candidate-item-historical-screen-v1'
assert diagnostic.CODE_VERSION == 'm3-clv-supplemental-candidate-set-precheck-v1'
assert str(Path(candidate_runner.__file__).resolve()).startswith(str(repo.resolve()))
assert str(Path(diagnostic.__file__).resolve()).startswith(str(repo.resolve()))

cfg = candidate_runner.configure_clv_candidate_item_supplemental_run(
    evaluation_authorized=False,
)
summary = candidate_runner.preflight_summary(cfg)
assert summary['seed'] == 42
assert summary['performance_evaluation_authorized'] is False
assert summary['historical_development_split']['train_end_inclusive'] == 683
assert summary['historical_development_split']['evaluation_start_inclusive'] == 684
assert summary['historical_development_split']['evaluation_end_inclusive'] == 690
assert summary['historical_development_split']['final_test_constructed'] is False
assert summary['historical_development_split']['holdout_constructed'] is False
assert summary['m3']['historical_clv_proxy'] == 'N_hat * V_hat'
assert summary['m3']['base_candidate_items'] == 100
assert summary['m3']['supplemental_candidate_items'] == 20
assert abs(summary['m3']['base_mass'] - 5 / 6) < 1e-12
assert abs(summary['m3']['supplemental_mass'] - 1 / 6) < 1e-12
assert summary['m3']['candidate_train_pairs_excluded'] is True
assert summary['m3']['gamma'] == 0.075
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['sample_weighting'] is False
assert summary['fixed']['min_item_interactions'] == 1
print(json.dumps(summary, ensure_ascii=False, indent=2))


## 3. 학습 없는 후보 집합 진단 실행

`310-284=26`이라는 순증가를 actual-only 26개로 해석하지 않고, actual-only·shuffle-only 정답을 분리합니다. actual-only 정답 중 일반 후보 밖의 정답이 실제로 남는지도 확인합니다.


In [ ]:
set_summary = diagnostic.run_supplemental_candidate_precheck(cfg)
RESULT_PATHS = set_summary.attrs['result_paths']
print(json.dumps(RESULT_PATHS, ensure_ascii=False, indent=2))


## 4. 전체·CLV 분위별 결과와 중단 판정


In [ ]:
from IPython.display import display

display(set_summary.sort_values(['clv_group', 'candidate_block', 'graph_arm']))
reading = set_summary.attrs['reading']
graph = set_summary.attrs['graph_diagnostics']
support = graph['supplemental_support']
assert support['base_edges_identical'] is True
assert support['base_extra_overlap'] == 0
assert support['train_pair_edges'] == 0
assert support['edges_per_active_user'] == 120
assert support['max_base_mass_error'] < 1e-6
assert support['max_extra_mass_error'] < 1e-6
print('\n후보 집합 판정:')
print(json.dumps(reading, ensure_ascii=False, indent=2))
print('\n그래프 불변조건:')
print(json.dumps(support, ensure_ascii=False, indent=2))
print('\n주의: precheck_passed=True여도 성능 개선을 뜻하지 않습니다.')
print('새 성능평가는 승인된 새 test 구간 또는 독립 데이터에서만 실행합니다.')
